# Kirby: run the development baseline in Colab
Target: **johnsnowlabs/JSL-MedPhi2-2.7B**, a public, ungated checkpoint verified on 2026-11-07. No Gemma access, JSL API key, or OpenAI key is needed for this baseline. The GPU path still needs validation; no improvement has been demonstrated. It does not edit weights or call Astra.

**Before running:**
1. Select **Runtime → Change runtime type → GPU**. The default **FP32** profile can be attempted on a **T4**, with roughly 11 GB for weights plus inference overhead. L4/A100 can also use **BF16** by explicitly changing `PRECISION` below. Actual memory needs must be measured.
2. Review the [JSL model card](https://huggingface.co/johnsnowlabs/JSL-MedPhi2-2.7B) and [model terms](https://huggingface.co/johnsnowlabs). CC BY-NC-ND restricts commercial use and distribution of modified weights; this notebook evaluates the original checkpoint and exports reports only.
3. `HF_TOKEN` is **optional**. Anonymous downloads are the default. If authentication is needed for Hub rate limits, add it in Colab Secrets and explicitly enable `USE_HF_TOKEN` below.
4. Have `kirby_colab.zip` and the official English four-option MedQA `train.jsonl`, `dev.jsonl`, and `test.jsonl` files ready. The original data release is linked at https://github.com/jind11/MedQA.

Run cells in order. Start with the **8-question development smoke check**. It is not a full benchmark. Keep the notebook open while work runs and download results afterward; Colab storage is temporary. GPU work consumes your available runtime/compute allocation.

## 1. Upload the backend bundle
Upload only `kirby_colab.zip`. The bundle contains source and tests, without frontend files, credentials, datasets, or model weights. Rerunning this cell creates a new workspace.

In [ ]:
from google.colab import files, userdata
from pathlib import Path, PurePosixPath
import io, json, os, re, stat, subprocess, sys, tempfile, zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only kirby_colab.zip in this cell.')
workspace = Path(tempfile.mkdtemp(prefix='kirby-', dir='/content'))
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    members = archive.infolist()
    if len(members) > 100 or sum(m.file_size for m in members) > 5_000_000:
        raise ValueError('Unexpected bundle size.')
    names = set()
    for member in members:
        name = PurePosixPath(member.filename)
        valid_source = (len(name.parts) == 3 and name.parts[0] == 'backend' and
                        ((name.parts[1] in {'kirby', 'tests'} and name.suffix == '.py') or
                         (name.parts[1] == 'docs' and name.suffix == '.md') or
                         (name.parts[1] == 'notebooks' and name.suffix == '.ipynb')))
        if (str(name) not in {'backend/pyproject.toml', 'backend/README.md'} and not valid_source):
            raise ValueError('Unexpected bundle member.')
        if ('..' in name.parts or name.is_absolute() or '\\' in member.filename or
                ':' in member.filename or stat.S_ISLNK(member.external_attr >> 16) or
                member.filename in names):
            raise ValueError('Unsafe or duplicate bundle member.')
        names.add(member.filename)
    archive.extractall(workspace)
del uploaded
backend = workspace / 'backend'
assert (backend / 'kirby' / 'cli.py').is_file(), 'Missing backend entry point.'
print('Backend workspace:', backend)


## 2. Select precision and check hardware
Use `float32` on T4. You may explicitly choose `bfloat16` on L4/A100 to reduce memory. The selected precision is frozen in each run; scores from different precisions are different experiments. There is no quantization or automatic precision fallback.

In [ ]:
MODEL_ID = 'johnsnowlabs/JSL-MedPhi2-2.7B'
MODEL_REVISION = 'b313a242eb211863f7b62624b345b321a3b114ea'
PRECISION = 'float32'  # T4: float32. L4/A100: optionally bfloat16.
assert PRECISION in ('float32', 'bfloat16')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime first.')
gpu = torch.cuda.get_device_properties(0)
print(f'GPU: {gpu.name}; VRAM: {gpu.total_memory / 2**30:.1f} GiB')
if PRECISION == 'bfloat16' and (gpu.major < 8 or not torch.cuda.is_bf16_supported()):
    raise RuntimeError('BF16 requires a supported GPU; explicitly select float32 for T4.')
print('Model:', MODEL_ID, '| precision:', PRECISION)
print('Actual peak memory will be recorded by the worker. This is not yet a GPU-validated configuration.')


## 3. Install the worker and run its tests
A separate Python environment reuses Colab's installed CUDA PyTorch while keeping backend library changes out of the notebook kernel. We use virtualenv's built-in creator and bundled pip seeder, so setup does not depend on the runtime's venv/ensurepip packages. Each retry uses a fresh directory and displays command output, including errors. This can take several minutes. Passing CPU tests does not validate the checkpoint.

In [ ]:
def run_setup(command, **kwargs):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kwargs)
    print(result.stdout, end='')
    result.check_returncode()

print('Installing the environment creator...')
bootstrap = Path(tempfile.mkdtemp(prefix='virtualenv-bootstrap-', dir=workspace))
run_setup([sys.executable, '-m', 'pip', 'install', '--target', str(bootstrap), 'virtualenv>=20.35,<21'])
bootstrap_env = os.environ.copy()
bootstrap_env['PYTHONPATH'] = str(bootstrap)
environment = Path(tempfile.mkdtemp(prefix='worker-venv-', dir=workspace))
run_setup([sys.executable, '-m', 'virtualenv', '--creator', 'builtin', '--seeder', 'app-data',
           '--no-periodic-update', '--system-site-packages', '--python', sys.executable, str(environment)], env=bootstrap_env)
worker_python = str(environment / 'bin' / 'python')
print('Installing the backend into the worker environment...')
run_setup([worker_python, '-m', 'pip', 'install', '-e', str(backend) + '[gpu]'])
run_setup([worker_python, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=backend)
run_setup([worker_python, '-c', 
    'import torch; from transformers import AutoTokenizer, PhiConfig, PhiForCausalLM; '
    'assert torch.cuda.is_available(); '
    'print("Worker GPU:", torch.cuda.get_device_name(0))'])


## 4. Verify the pinned public revision
No token is required by default. This cell checks the exact commit, without following a moving branch. If `USE_HF_TOKEN` is enabled, the token comes from Colab Secrets and is never printed or written to reports. The remaining weight download occurs during the baseline run.

In [ ]:
USE_HF_TOKEN = False
if USE_HF_TOKEN:
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        raise RuntimeError('Add HF_TOKEN in Colab Secrets and enable notebook access.') from None
    os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'
else:
    os.environ.pop('HF_TOKEN', None)
    os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '1'
os.environ['HF_HOME'] = '/content/kirby-hf-cache'
revision = subprocess.check_output([worker_python, '-c',
    'import sys; from huggingface_hub import HfApi; print(HfApi().model_info(sys.argv[1], revision=sys.argv[2]).sha)',
    MODEL_ID, MODEL_REVISION], text=True).strip()
if revision != MODEL_REVISION or not re.fullmatch(r'[0-9a-f]{40}', revision):
    raise ValueError('Provider revision differs from the selected immutable commit.')
(workspace / 'model-revision.txt').write_text(revision, encoding='utf-8')
print('Pinned JSL revision:', revision)


## 5. Upload the three official MedQA splits
Upload exactly `train.jsonl`, `dev.jsonl`, and `test.jsonl` from the English four-option release. Preparation will check all splits for overlap; the baseline worker reads only development questions. Repeated option texts are preserved and recorded in source metadata, with scoring still using the supplied answer_idx. A failed validation needs investigation, not silent filtering.

In [ ]:
data = backend / 'data' / 'medqa'
data.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
expected = {'train.jsonl', 'dev.jsonl', 'test.jsonl'}
if set(uploaded) != expected:
    raise ValueError('Select exactly train.jsonl, dev.jsonl, and test.jsonl.')
for name, payload in uploaded.items():
    target = data / name
    if target.exists() and target.read_bytes() != payload:
        raise ValueError('Existing source differs. Create a new workspace for a changed dataset.')
    target.write_bytes(payload)
del uploaded
print('Dataset files ready for validation.')


## 6. Run an 8-question smoke check
The first run downloads several GB of model files. Keep `DEVELOPMENT_COUNT = 8` until this completes and its outputs are inspected. Every run also makes two unchanged calibration passes.
Profile v2 accepts a letter with an optional period; JSL generation stops at a complete label such as `D.`. Raw output and the stop policy are recorded. This is a new baseline protocol, not a rescore of previous failed runs.

After a successful smoke check, change `DEVELOPMENT_COUNT` to `256` and rerun this cell for the guide's development sample. It creates a separate plan. Neither run evaluates final questions or proves improvement. The worker's cooperative time limit includes loading; raise it only as needed based on measured time. The parent notebook does not hold model weights, and each worker exits to release its GPU memory.

In [ ]:
DEVELOPMENT_COUNT = 8  # Change to 256 after inspecting a successful smoke run.
MAX_SECONDS = 3600
assert DEVELOPMENT_COUNT in (8, 256)
plan = backend / 'data' / f'plan-dev-{DEVELOPMENT_COUNT}'
if not plan.exists():
    run_setup([worker_python, '-m', 'kirby', 'prepare',
        '--train', str(data / 'train.jsonl'), '--dev', str(data / 'dev.jsonl'),
        '--test', str(data / 'test.jsonl'), '--output', str(plan),
        '--development-count', str(DEVELOPMENT_COUNT)], cwd=backend)
run_setup([worker_python, '-m', 'kirby', 'validate', '--manifest', str(plan / 'development.json')], cwd=backend)
runs = backend / 'runs'
before = set(runs.glob('*/report.json'))
label = 'smoke check' if DEVELOPMENT_COUNT == 8 else 'development baseline'
print(f'Starting {label}: {DEVELOPMENT_COUNT} scored questions. No final evaluation.')
process = subprocess.Popen([worker_python, '-u', '-m', 'kirby', 'baseline',
    '--manifest', str(plan / 'development.json'), '--model', MODEL_ID,
    '--revision', revision, '--precision', PRECISION,
    '--allow-download', '--max-seconds', str(MAX_SECONDS), '--output', str(runs)],
    cwd=backend, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
except KeyboardInterrupt:
    import signal
    process.send_signal(signal.SIGINT)
    try:
        remaining, _ = process.communicate(timeout=20)
    except subprocess.TimeoutExpired:
        process.kill()
        remaining, _ = process.communicate()
    print(remaining, end='', flush=True)
    raise
finally:
    process.stdout.close()
created = set(runs.glob('*/report.json')) - before
if len(created) != 1:
    raise RuntimeError('No unique run report was created. Inspect the worker output.')
report_path = created.pop()
report = json.loads(report_path.read_text(encoding='utf-8'))
print(json.dumps({'scope': label, 'state': report['state'], 'outcome': report['outcome'],
                  'calibration': report.get('calibration'),
                  'development': report['development'], 'error': report.get('error')}, indent=2))
print('Report:', report_path)
if return_code:
    calibration_path = report_path.parent / 'calibration.jsonl'
    if calibration_path.exists():
        print('Calibration first-pass outputs (not development scores):')
        for line in calibration_path.read_text(encoding='utf-8').splitlines():
            record = json.loads(line)
            if record['pass'] == 1:
                print({key: record[key] for key in ('text', 'generated_tokens', 'truncated', 'valid')})
    print('Run incomplete. Download its evidence and investigate before proceeding.')


## 7. Download the evidence before ending the session
Run this even after an incomplete experiment. The download includes run reports, raw predictions, calibration outputs, events, frozen plans, revision, source code, and installed package versions. It excludes credentials, the raw benchmark, and model weights. Reuse the same benchmark source files when reproducing these results. An abrupt Colab runtime deletion can lose files before this cell runs.

In [ ]:
from datetime import datetime, timezone
inventory = subprocess.check_output([worker_python, '-m', 'pip', 'list', '--format=json'], text=True)
(workspace / 'package-versions.json').write_text(inventory, encoding='utf-8')
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
destination = workspace / f'kirby-evidence-{stamp}.zip'
with zipfile.ZipFile(destination, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for source in sorted((backend / 'runs').glob('*/*')):
        if source.is_file() and source.suffix in {'.json', '.jsonl', '.md'}:
            archive.write(source, str(source.relative_to(workspace)))
    for source in sorted((backend / 'data').glob('plan-dev-*/*.json')):
        archive.write(source, str(source.relative_to(workspace)))
    for source in sorted((backend / 'kirby').glob('*.py')):
        archive.write(source, str(source.relative_to(workspace)))
    for source in (backend / 'pyproject.toml', workspace / 'package-versions.json', workspace / 'model-revision.txt'):
        if source.is_file():
            archive.write(source, str(source.relative_to(workspace)))
print('Downloading:', destination.name)
files.download(str(destination))


## What comes next
Keep the exported evidence. A successful smoke check validates only a small execution path. A completed 256-question run establishes a development baseline under this recorded protocol. Continue with `kirby_next_steps.ipynb` to repeat the baseline and verify controlled rollback, then `kirby_astra_colab.ipynb` for one bounded Astra proposal. Regression evaluation, retained candidates, and export/reload remain pending backend work.

The model has a 2,048-token context. Oversized questions stop the run instead of being truncated. JSL's terms restrict distribution of modified checkpoints; a future edited-model release needs a compatible license or separate permission.

References: [Colab availability and lifetime limits](https://research.google.com/colaboratory/faq.html), [JSL model](https://huggingface.co/johnsnowlabs/JSL-MedPhi2-2.7B), [JSL terms](https://huggingface.co/johnsnowlabs), [MedQA source](https://github.com/jind11/MedQA), [Hugging Face environment variables](https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables).